# 1.1 — Trevas Jupyter introduction

Welcome to the **Trevas Jupyter** training track.

**In this notebook**
- What Trevas Jupyter is and how it works
- The [Trevas](https://github.com/Making-Sense-Info/Trevas) open-source VTL engine
- All **add-on functions** available in notebooks (load, inspect, export, SDMX)

**Next:** [1.2-vtl-intro.ipynb](1.2-vtl-intro.ipynb) — VTL language concepts and information model.

> Select the **Trevas VTL** kernel for every code cell in this folder.

## What is Trevas Jupyter? <img style="float: right;" src="images/VTL_9.png" width="30" height="30">

**Trevas Jupyter** is a JupyterLab environment where each notebook cell runs **VTL** (Validation & Transformation Language) through the **Trevas** engine.

```
┌──────────────┐     ┌─────────────────┐     ┌──────────────────┐
│  JupyterLab  │ ──▶ │  Trevas VTL     │ ──▶ │  Apache Spark    │
│  (notebook)  │     │  kernel         │     │  (distributed    │
└──────────────┘     └─────────────────┘     │   execution)     │
                                              └──────────────────┘
```

| Layer | Role |
|---|---|
| **JupyterLab** | Interactive notebooks, file browser, rich outputs |
| **Trevas kernel** | Parses VTL, evaluates transformations, renders tables |
| **Trevas engine** | Standard-compliant VTL 2.x interpreter ([Making-Sense-Info/Trevas](https://github.com/Making-Sense-Info/Trevas)) |
| **Spark** | Scales data loading and transformations to large volumes |

The **jupyterlab-vtl-2-1** extension adds syntax highlighting, linting, and autocomplete for VTL in the editor.

Docker image: [`makingsenseinfo/trevas-jupyter`](https://hub.docker.com/r/makingsenseinfo/trevas-jupyter) — also used on platforms like [Onyxia](https://datalab.sspcloud.fr/).

## The Trevas engine

[**Trevas**](https://github.com/Making-Sense-Info/Trevas) is an **open-source** implementation of the SDMX **VTL 2.x** standard. It is designed for statistical and data-production workflows:

- **Standard operators** — `keep`, `drop`, `join`, `aggregate`, … with defined semantics
- **Spark backend** — same VTL script, from laptop to cluster
- **Extensible** — Trevas Jupyter registers **custom functions** for I/O and notebook ergonomics

You write **declarative** transformation schemes; Trevas derives execution order and runs them on Spark datasets.

> VTL describes *what* to compute. Trevas + Spark handle *how* to run it efficiently.

## How a notebook cell works

1. Write one or more VTL **statements** in a code cell (`name := expression;`).
2. Run the cell — the Trevas kernel evaluates the **transformation scheme**.
3. Results appear below: tables (`show`), metadata (`showMetadata`), or text (`size`).
4. Re-running a cell clears variables assigned **in that cell** first — safe for iterative exploration.

**Important:** `show` and `showMetadata` return display objects — always **assign** them so Jupyter renders the output:

```vtl
preview := show(my_dataset);
meta    := showMetadata(my_dataset);
```

Bare calls like `show(my_dataset);` may not display correctly.

## Add-on functions — overview

Standard VTL does not define file I/O or notebook display. Trevas Jupyter adds the following functions ([full reference](https://github.com/Making-Sense-Info/Trevas-Jupyter#custom-functions)):

### Data loading & export

| Function | Arguments | Returns | Description |
|---|---|---|---|
| `loadCSV` | `url` | Dataset | Load a CSV file |
| `loadParquet` | `url` | Dataset | Load a Parquet file |
| `loadSas` | `url` | Dataset | Load a SAS data set |
| `writeCSV` | `(url, dataset)` | String | Write a data set to CSV |
| `writeParquet` | `(url, dataset)` | String | Write a data set to Parquet |

### Inspection

| Function | Arguments | Returns | Description |
|---|---|---|---|
| `show` | `dataset` | DisplayData | Table preview (first rows) |
| `showMetadata` | `dataset` | DisplayData | Structure: names, types, **roles** |
| `size` / `getSize` | `dataset` | String | Row count |

### SDMX workflows

| Function | Arguments | Returns | Description |
|---|---|---|---|
| `loadSDMXEmptySource` | `(sdmxMesUrl, structureId)` | Dataset | Empty SDMX source |
| `loadSDMXSource` | `(sdmxMesUrl, structureId, dataUrl)` | Dataset | SDMX source with data |
| `runSDMXPreview` | `sdmxMesUrl` | DisplayData | Run VTL transformations on empty sources |
| `runSDMX` | `(sdmxMesUrl, dataLocations)` | DisplayData | Run VTL transformations with data |
| `getTransformationsVTL` | `sdmxMesUrl` | String | Extract VTL transformations from SDMX message |
| `getRulesetsVTL` | `sdmxMesUrl` | String | Extract VTL rulesets from SDMX message |

### Data locations

Paths can be **local** (`./data/file.csv`), **absolute** (`/home/jovyan/work/…`), **HTTPS**, or **S3** (`s3://bucket/path` → mapped to `s3a://` for Spark).

CSV options (`delimiter`, `quote`, `header`, …) are passed as **URL-encoded query parameters**.

## `show` and `showMetadata` <img style="float: right;" src="images/VTL_8.png" width="30" height="30">

| Function | Use when you need… |
|---|---|
| `show(ds)` | Row-level **values** (data points) |
| `showMetadata(ds)` | **Structure** — component names, types, identifier/measure/attribute roles |

Always assign the result to a variable name.

In [ ]:
// Load training data (comma-separated)
grades_raw <- loadCSV("data/students_grades.csv?delimiter=,");

// Inspect structure before assigning roles
raw_meta := showMetadata(grades_raw);

In [ ]:
// Attach roles, then inspect again
grades := grades_raw
  [calc identifier Student_Id := Student_Id,
        identifier Subject := Subject,
        identifier Semester := Semester,
        measure Grade := Grade,
        attribute School := School,
        attribute Teacher := Teacher];

structured_meta := showMetadata(grades);
preview         := show(grades);

## `loadCSV` <img style="float: right;" src="images/VTL_27.png" width="30" height="80">

Default CSV options: delimiter **`;`**, quote **`"`**, header **`true`**.

| Example | Meaning |
|---|---|
| `loadCSV("data/ds1.csv")` | Semicolon-separated (default) |
| `loadCSV("data/ds2.csv?delimiter=%2c&quote=%27")` | Comma + single-quote |
| `loadCSV("data/students_grades.csv?delimiter=,")` | Comma-separated |
| `loadCSV("s3://bucket/data.csv?delimiter=%7C")` | Pipe delimiter on S3 |

In [ ]:
// Semicolon-separated (Trevas default)
cities_raw <- loadCSV("data/ds1.csv");
ds1_meta   := showMetadata(cities_raw);

// Comma-separated, single-quoted fields
nuts_raw <- loadCSV("data/ds2.csv?delimiter=%2c&quote=%27");
ds2_meta := showMetadata(nuts_raw);

## `writeCSV` and `writeParquet`

Persist a data set to disk (or S3). Spark writes a **directory** of part files, not always a single file.

```vtl
written := writeCSV("./output/my_table", my_dataset);
written := writeParquet("./output/my_table", my_dataset);
```

CSV write options use the same URL query syntax as `loadCSV` (e.g. `?delimiter=%7C&quote=%27`).

## `size` / `getSize`

Return the number of **data points** (rows) in a data set:

```vtl
n_rows := size(grades);
// alias:
n_rows := getSize(grades);
```

In [ ]:
row_count := getSize(grades);

## Other loaders (reference)

```vtl
// Parquet
parquet_ds <- loadParquet("./data/my_table.parquet");

// SAS
sas_ds <- loadSas("./data/my_table.sas7bdat");

// SDMX — see 2-SDMX-Sample/ in this repo for a full example
empty <- loadSDMXEmptySource("path/to/message.xml", "STRUCTURE_ID");
source <- loadSDMXSource("message.xml", "STRUCT_ID", "data.csv");
preview := runSDMXPreview("message.xml");
result  := runSDMX("message.xml", "STRUCT_ID,data.csv");
vtl_txt := getTransformationsVTL("message.xml");
rules   := getRulesetsVTL("message.xml");
```

SDMX data sources are currently expected to be **CSV** files.

## Training path

| Step | Notebook | Topic |
|---|---|---|
| **1.1** (this file) | [1.1-trevas-intro.ipynb](1.1-trevas-intro.ipynb) | Trevas Jupyter & add-on functions |
| **1.2** | [1.2-vtl-intro.ipynb](1.2-vtl-intro.ipynb) | VTL concepts & information model |
| **2** | [2-core-rules.ipynb](2-core-rules.ipynb) | Assignments, comments, roles |
| **3** | [3-first-operators.ipynb](3-first-operators.ipynb) | `keep`, `drop`, `rename`, `writeCSV` |

---

Continue with **[1.2-vtl-intro.ipynb](1.2-vtl-intro.ipynb)**.